# Скоринг горнолыжных курортов

## Оглавление
1. [Расчет баллов для ранжирования горнолыжных курортов](#Расчет-баллов-для-ранжирования-горнолыжных-курортов)
2. [Итоговый рейтинг](#Итоговый-рейтинг)
3. [Сценарный рейтинг](#Сценарный-рейтинг)
4. [Ответы на вопросы для исследования согласно ТЗ](#Ответы-на-вопросы-для-исследования-согласно-ТЗ)

## Расчет баллов для ранжирования горнолыжных курортов

### Подготовка окружения

In [1]:
import pandas as pd
import numpy as np

mart = pd.read_csv('../data/mart/resort_features.csv')
print(mart.shape)
mart.head(3)

(32, 43)


,resort_id,name_ru,region,latitude,longitude,coordinate_quality,operating,status_note,piste_total_km_6km,lift_lines_6km,...,freeride,tubing,adult_price_min_rub,adult_price_max_rub,child_price_min_rub,child_price_max_rub,trips_2025,trips_growth_2025,nights_2025,avg_stay_nights_2025
0,R001,Роза Хутор,Краснодарский край,43.646390,40.332220,checked_ok,yes,NaN,159.4,64,...,1.0,1.0,7920.0,192000.0,4000.0,96000.0,18427326.0,-0.038392,110670767,6.005796
1,R002,Красная Поляна,Краснодарский край,43.677244,40.262795,official_verified,yes,NaN,93.2,58,...,1.0,1.0,3900.0,5250.0,2300.0,3050.0,18427326.0,-0.038392,110670767,6.005796
2,R003,Газпром Поляна,Краснодарский край,43.699608,40.313290,official_verified,yes,NaN,73.5,46,...,1.0,1.0,4100.0,4100.0,2550.0,2550.0,18427326.0,-0.038392,110670767,6.005796


### Словарь скоринга
Ранжирование будет основываться на двух категориях показателей.

#### Базовые показатели и веса согласно ТЗ для составления базового рейтинга

| Показатель | Колонка витрины | Шкала | Почему такая шкала | Вес ТЗ |
|---|---|---|---|---|
| Поездки по региону 2025 | trips_2025 | log min max | Логарифм уменьшает влияние единичных очень больших значений | 25 |
| Рост поездок | trips_growth_2025 | Линейная шкала с ограничением | Формула не зависит от выборки | 10 |
| Средняя длительность | avg_stay_nights_2025 | Линейная шкала с ограничением | Формула не зависит от выборки | 5 |
| Трассы OSM | piste_total_km_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 10 |
| Подъемники OSM | lift_lines_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 7 |
| Размещение OSM | lodging_objects_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 5 |
| Питание и места интереса OSM | food_objects_6km + attraction_objects_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 3 |
| Снежные дни | avg_snow_days_per_season | min max | Обычная нормализация | 8 |
| Морозные дни | avg_freezing_days_per_season | min max | Обычная нормализация | 6 |
| Сезонный снегопад | avg_snowfall_cm_per_season | log min max | Логарифм уменьшает влияние единичных очень больших значений | 6 |
| Расстояние до аэропорта | nearest_airport_distance_km | Обратная log min max  | Чем меньше значение, тем лучше | 8 |
| Локальный транспорт OSM | transport_objects_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 2 |
| Минимальная разовая ски пасса (*было принято решение не использовать среднее) | adult_price_min_rub | Обратная log min max  | Чем меньше значение, тем лучше | 5 |


#### Сценарные показатели для составления опционального рейтинга

| Показатель | Колонка витрины | Шкала | Почему такая шкала |
|---|---|---|---|
| Солнечные часы | winter_mean_sunshine_hours | порог 6 | 6 часов солнца в зимний день = комфортно | 
| Расстояние до жд-станции | nearest_railway_station_distance_km | Обратная log min max  | ближе станция - проще добраться | 
| Зеленые трассы | green | порог 10 | 10 зеленых = достаточный выбор для новичков | 
| Синие трассы | blue | порог 15 | синих у всех много, "достаточно" больше |
| Красные трассы | red | порог 10 | средний вариант |
| Черные трассы | black | порог 5 | черные - редкость, 5 = серьезная гора |
| Наличие фрирайда | freeride | 0/1 × 100 | бинарный признак: есть или нет |
| Наличие тюбинг трассы | tubing | 0/1 × 100 | бинарный признак: есть или нет |
| Перепад высот | drop_m | порог 850 | 850 м = 85-й перцентиль, «достаточно эпично»; наш порог по правилу насыщения |


### Подготовка функций для расчета баллов
* Скоринг: сырые показатели → баллы 0–100.
* Вход: витрина data/mart/resort_features.csv (32 курорта).

#### Шкала min max
* Минимум выборки получает 0, максимум - 100.
* Используется для снежных и морозных дней.
* Тестируем на условных числах, прежде чем пускать на курорты.

In [2]:
# Суть: кладем колонку сырых чисел - получаем колонку баллов 0-100
def min_max_score(column):
    x_min = column.min()
    x_max = column.max()
    return 100 * (column - x_min) / (x_max - x_min)


# Тест на условных числах: одно крошечное, одно среднее, одно гигантское.
test = pd.Series([0, 10, 100, 1000])
min_max_score(test)

0      0.0
1      1.0
2     10.0
3    100.0
dtype: float64

#### Шкала log min max
* Логарифм уменьшает влияние единичных очень больших значений.
* Ноль допустим, потому что перед логарифмом к показателю прибавляется единица.
* Сначала пропускаем числа через ln(1+x) - он сжимает гигантов.
* Затем та же функция, что и для min max.
* Используется для поездок, OSM объектов, локального транспорта и сезонного снегопада.

In [3]:
# ln(1+x): единица внутри, чтобы ноль не ломал логарифм (ln(0) не существует)
def log_min_max_score(column):
    log_values = np.log(1 + column)
    return min_max_score(log_values)

# Тот же тест на тех же числах
log_min_max_score(test)

0      0.000000
1     34.708068
2     66.801047
3    100.000000
dtype: float64

#### Шкала обратная log min max
* Для признаков, где меньше = лучше.
* Сначала обычный логарифмический балл, потом разворот.
* Первый становится последним.
* Используется для расстояния до аэропорта и жд-вокзала и цены.

In [4]:
# Обратная шкала: берем готовый лог-балл и разворачиваем его
def inverse_score(column):
    return (100 - log_min_max_score(column)).clip(0, 100)

# Тот же тест: ряд как будто "расстояния в км" - кто ближе, тот лучше
inverse_score(test)

0    100.000000
1     65.291932
2     33.198953
3      0.000000
dtype: float64

#### Линейная шкала с ограничением
* Для роста и ночевок ТЗ дает готовые формулы с обрезкой краев: рост −25% и хуже = 0, +25% и выше = 100, нулевой рост = 50; ночи: 1 ночь = 0, 7 ночей и больше = 100.
* Эти формулы не зависят от выборки.
* Используется для роста поездок и ночевок.

In [5]:
# clip(0, 100) обрезает все, что вылезло за края, - страховка от аномалий.
def growth_score(growth):
    return (50 + 200 * growth).clip(0, 100)

def stay_score(nights):
    return ((nights - 1) / 6 * 100).clip(0, 100)

# Тест роста: впадина, граница, Краснодарский край, ноль, потолок, аномалия
test_growth = pd.Series([-0.5, -0.25, -0.0384, 0, 0.25, 0.86])
print(growth_score(test_growth))

# Тест ночевок: минимум, медиана наших данных, Краснодарский край, потолок
test_stay = pd.Series([1, 3.56, 6.006, 7.83])
print(stay_score(test_stay))

0      0.00
1      0.00
2     42.32
3     50.00
4    100.00
5    100.00
dtype: float64
0      0.000000
1     42.666667
2     83.433333
3    100.000000
dtype: float64


#### Пороговая шкала
* Переводит значение в баллы не сравнением с другими курортами, а сравнением с уровнем "достаточно хорошо": дошел до порога - 100, не дошел - доля пути.
* Перевыполнение не добавляет баллов.
* Используется для сценарных показателей: сложность трасс, солнечные часы, перепад высот.

In [6]:
def threshold_score(column, threshold):
    return (100 * column / threshold).clip(0, 100)

# Тест на контрольных точках: зеленые трассы, порог = 10
# 0 - трасс нет, 2 - медиана наших зеленых, 5 - половина порога
# 10 - сам порог, 16 - Манжерок
threshold_score(pd.Series([0, 2, 5, 10, 16]), 10)
# ожидание: 0 / 20 / 50 / 100 / 100

0      0.0
1     20.0
2     50.0
3    100.0
4    100.0
dtype: float64

### Применение функций для расчета баллов
* Суффикс _score означает, что это уже балл, не сырые данные.
* В заголовках блоков - функция, в скобках - смысловая группа показателей из ТЗ.

#### Шкала min-max
* Ровные величины без гигантов.
* Снежные и морозные дни: разброс ровный (25–107 и 27–180) - обычная нормализация.

In [7]:
# scores - новая таблица: только курорт и баллы, сырые числа остаются в mart
scores = pd.DataFrame({
    'resort_id': mart['resort_id'],
    'name_ru': mart['name_ru'],
    'snow_days_score': min_max_score(mart['avg_snow_days_per_season']),
    'frost_days_score': min_max_score(mart['avg_freezing_days_per_season']),
})

print(scores.shape)
scores.head()

(32, 4)


,resort_id,name_ru,snow_days_score,frost_days_score
0,R001,Роза Хутор,9.535452,4.681404
1,R002,Красная Поляна,0.000000,0.000000
2,R003,Газпром Поляна,37.652812,27.178153
3,R004,Шерегеш,83.618582,83.355007
4,R005,Архыз,47.188264,33.029909


##### Проверка: шкала min-max
* Все баллы блока в 0-100.
* Пропусков нет - климат посчитан для всех 32.
* Санити-чек: кто самый снежный.

In [8]:
num = scores.select_dtypes('number')
print('max:', num.max().max(), '/ min:', num.min().min())
print('NaN:', num.isna().sum().sum(), '| колонок:', scores.shape[1])

# кто самый снежный
scores.sort_values('snow_days_score', ascending=False).head(3)

max: 100.0 / min: 0.0
NaN: 0 | колонок: 4


,resort_id,name_ru,snow_days_score,frost_days_score
12,R013,Большой Вудъявр,100.000000,100.000000
13,R014,Кукисвумчорр,97.066015,91.157347
5,R006,Домбай,95.110024,51.235371


#### Шкала log min max
* "Питание и места интереса" - одной суммой, потому что в ТЗ у группы один вес.

In [9]:
# Региональный спрос
scores['trips_score'] = log_min_max_score(mart['trips_2025'])

# Инфраструктура
scores['trails_km_score'] = log_min_max_score(mart['piste_total_km_6km'])
scores['lifts_score'] = log_min_max_score(mart['lift_lines_6km'])
scores['lodging_score'] = log_min_max_score(mart['lodging_objects_6km'])
scores['food_score'] = log_min_max_score(mart['food_objects_6km'] + mart['attraction_objects_6km'])

# Сезонный снегопад и локальный транспорт
scores['snowfall_score'] = log_min_max_score(mart['avg_snowfall_cm_per_season'])
scores['transport_score'] = log_min_max_score(mart['transport_objects_6km'])

print(scores.shape)
scores.head()

(32, 11)


,resort_id,name_ru,snow_days_score,frost_days_score,trips_score,trails_km_score,lifts_score,lodging_score,food_score,snowfall_score,transport_score
0,R001,Роза Хутор,9.535452,4.681404,95.844308,100.000000,100.000000,80.650867,82.750048,31.800561,62.447048
1,R002,Красная Поляна,0.000000,0.000000,95.844308,86.512562,97.217950,100.000000,100.000000,7.293903,72.089495
2,R003,Газпром Поляна,37.652812,27.178153,95.844308,80.567172,90.686087,96.041159,91.540557,69.087693,64.500308
3,R004,Шерегеш,83.618582,83.355007,39.253523,91.033834,71.379927,98.518906,63.633493,64.839101,65.738985
4,R005,Архыз,47.188264,33.029909,39.869439,83.578282,55.897040,94.566116,72.599951,65.583847,48.302554


##### Проверка: шкала log min max
* Все баллы блока в 0-100.
* Пропусков нет - OSM и спрос посчитаны для всех 32.
* Санити-чек: кто чемпион по снегопаду.

In [10]:
num = scores.select_dtypes('number')
print('max:', num.max().max(), '/ min:', num.min().min())
print('NaN:', num.isna().sum().sum(), '| колонок:', scores.shape[1])

# чемпион по снегопаду - смотрим и его сырые см в витрине
scores.sort_values('snowfall_score', ascending=False).head(3)

max: 100.0 / min: 0.0
NaN: 0 | колонок: 11


,resort_id,name_ru,snow_days_score,frost_days_score,trips_score,trails_km_score,lifts_score,lodging_score,food_score,snowfall_score,transport_score
5,R006,Домбай,95.110024,51.235371,39.869439,51.003882,55.89704,80.442088,64.947133,100.000000,29.393226
6,R007,Эльбрус,90.709046,95.448635,11.540457,61.132473,55.89704,76.201065,71.229156,91.283421,49.944391
7,R008,Чегет,88.264059,87.386216,11.540457,56.306472,51.46900,82.048411,75.680295,88.706640,49.144004


#### Шкала обратная log min max
* Аэропорт и цена - базовые показатели; расстояние до ЖД-станции - сценарный (в базовый рейтинг не входит, живет в отдельной таблице scenario_scores - создаем ее здесь: это первый сценарный балл по ходу функций).
* Цена ориентируется на минимальную разовую взрослую цену.
* Особые случаи цены - пояснение ниже.

In [11]:
# Доступность: ближе аэропорт - больше балл
scores['airport_score'] = inverse_score(mart['nearest_airport_distance_km'])

# Цена: дешевле = лучше. Ориентируемся на минимальную взрослую цену
scores['price_score'] = inverse_score(mart['adult_price_min_rub'])

# Сценарный: расстояние до ЖД-станции (ближе - лучше)
scenario_scores = pd.DataFrame({
    'resort_id': mart['resort_id'],
    'name_ru': mart['name_ru'],
    'railway_score': inverse_score(mart['nearest_railway_station_distance_km']),
})

# Восьми горным курортам станция не нашлась в радиусе сбора 50 км:
# это "дальше 50 км", а не пропуск - честный балл 0
scenario_scores['railway_score'] = scenario_scores['railway_score'].fillna(0)

print(scores.shape, scenario_scores.shape)
scores.head()

(32, 13) (32, 3)


,resort_id,name_ru,snow_days_score,frost_days_score,trips_score,trails_km_score,lifts_score,lodging_score,food_score,snowfall_score,transport_score,airport_score,price_score
0,R001,Роза Хутор,9.535452,4.681404,95.844308,100.000000,100.000000,80.650867,82.750048,31.800561,62.447048,58.268444,0.000000
1,R002,Красная Поляна,0.000000,0.000000,95.844308,86.512562,97.217950,100.000000,100.000000,7.293903,72.089495,59.997984,17.886005
2,R003,Газпром Поляна,37.652812,27.178153,95.844308,80.567172,90.686087,96.041159,91.540557,69.087693,64.500308,56.071628,16.623429
3,R004,Шерегеш,83.618582,83.355007,39.253523,91.033834,71.379927,98.518906,63.633493,64.839101,65.738985,21.385968,100.000000
4,R005,Архыз,47.188264,33.029909,39.869439,83.578282,55.897040,94.566116,72.599951,65.583847,48.302554,27.190474,28.611971


##### Проверка: шкала обратная log min max
* Все баллы обеих таблиц в 0-100.
* Ожидаем 3 пропуска в price_score - курорты без разовой цены (особые случаи ниже).
* railway_score пропусков не имеет: "нет станции в 50 км" уже переведено в честный 0.
* Санити-чек: кто ближе всех к аэропорту.

In [12]:
num = scores.select_dtypes('number')
print('max:', num.max().max(), '/ min:', num.min().min())
print('NaN:', num.isna().sum().sum(), '| колонок scores:', scores.shape[1])
print('ЖД = 0 у', (scenario_scores['railway_score'] == 0).sum(), 'курортов')

# кто ближе всех к аэропорту
scores.sort_values('airport_score', ascending=False).head(3)

max: 100.0 / min: 0.0
NaN: 3 | колонок scores: 13
ЖД = 0 у 9 курортов


,resort_id,name_ru,snow_days_score,frost_days_score,trips_score,trails_km_score,lifts_score,lodging_score,food_score,snowfall_score,transport_score,airport_score,price_score
15,R016,Горный воздух,65.770171,82.834850,11.967221,67.848401,39.821854,71.356424,90.434438,53.659263,91.840274,100.000000,91.912656
17,R018,Морозная гора,92.420538,88.686606,20.284219,23.055152,26.320814,19.840093,14.563993,84.176419,44.424674,95.160815,96.858991
30,R031,Красная Глинка,41.809291,55.396619,48.045044,19.148913,43.205219,46.321014,58.255974,19.477835,70.826975,91.186787,96.858991


##### Мини-чек цены

Топ-5 дорогих по цене входа - глазами смотрим, дневные это или абонементы.

In [13]:
mart.sort_values('adult_price_min_rub', ascending=False).head(5)[['name_ru', 'adult_price_min_rub', 'adult_price_max_rub']]

,name_ru,adult_price_min_rub,adult_price_max_rub
0,Роза Хутор,7920.0,192000.0
6,Эльбрус,6000.0,132000.0
23,Солнечная долина,5200.0,44000.0
8,Мамисон,4800.0,132000.0
5,Домбай,4400.0,12750.0


##### Особые случаи: цена
Пропуск цены → балл 50. У троих курортов нет цен (Телецкий, Аджигардак - реконструкция, КолаСпортЛэнд - закрыт): по ТЗ §12 их ценовой балл = 50 (нейтральная середина), а сам факт пропуска останется виден в витрине (там колонка adult_price_min_rub пустая) и потом в quality_flags.

In [14]:
scores['price_score'] = scores['price_score'].fillna(50)

# Кто именно получил нейтраль - проверяем по витрине, там честный пропуск еще виден
mart.loc[mart['adult_price_min_rub'].isna(), ['resort_id', 'name_ru', 'operating']]

,resort_id,name_ru,operating
11,R012,Телецкий,reconstruction
14,R015,КолаСпортЛэнд,closed
22,R023,Аджигардак,reconstruction


In [15]:
# Режим ТЗ: средняя двух границ. Абонементы в max тащат ее вверх - видно будет сразу
avg_price = (mart['adult_price_min_rub'] + mart['adult_price_max_rub']) / 2
scores['price_score_tz'] = inverse_score(avg_price).fillna(50)

compare = scores[['name_ru', 'price_score', 'price_score_tz']].sort_values('price_score', ascending=False)
print(compare.head(8))
compare.tail(8)

           name_ru  price_score  price_score_tz
3          Шерегеш   100.000000       25.481053
17   Морозная гора    96.858991       77.446731
30  Красная Глинка    96.858991       34.178246
13    Кукисвумчорр    92.777132       56.918099
29        Сорочаны    92.777132      100.000000
15   Горный воздух    91.912656       99.083361
21         Евразия    87.167323       40.529901
16         Холдоми    78.699295       49.276877


,name_ru,price_score,price_score_tz
25,Абзаково,28.121805,48.029709
1,Красная Поляна,17.886005,52.814571
2,Газпром Поляна,16.623429,54.691441
5,Домбай,14.840574,42.057258
8,Мамисон,12.643791,6.497326
23,Солнечная долина,10.622912,24.009969
6,Эльбрус,7.009895,6.347759
0,Роза Хутор,0.000000,0.000000


##### Решение по цене
* Трое с баллом 50 - неработающие курорты (Телецкий и Аджигардак в реконструкции, КолаСпортЛэнд закрыт): в сценарном показе и на дашборде они не участвуют, балл 50 живет только в контрольной базовой таблице, а пропуск остается виден в витрине и станет флагом missing price.
* Режим ТЗ отклонен: его порядок зависит от попадания абонемента в max (Мамисон с дневным 4800 упал на 6.5).
* Решение по цене: базовый ценовой признак - минимальная разовая цена взрослого билета; курортам без разовой цены вписана последняя известная 25/26.

#### Линейная шкала с ограничением
* Рост: −25%..+25% (формула 50 + 200×рост, клип 0-100).
* Длительность: от 1 до 7 ночей ((ночи − 1)/6).
* Границы заданы смыслом, а не выборкой.

In [16]:
# Рост поездок и средняя длительность
scores['growth_score'] = growth_score(mart['trips_growth_2025'])
scores['stay_score'] = stay_score(mart['avg_stay_nights_2025'])

print(scores.shape)
scores.head()

(32, 16)


,resort_id,name_ru,snow_days_score,frost_days_score,trips_score,trails_km_score,lifts_score,lodging_score,food_score,snowfall_score,transport_score,airport_score,price_score,price_score_tz,growth_score,stay_score
0,R001,Роза Хутор,9.535452,4.681404,95.844308,100.000000,100.000000,80.650867,82.750048,31.800561,62.447048,58.268444,0.000000,0.000000,42.321531,83.429939
1,R002,Красная Поляна,0.000000,0.000000,95.844308,86.512562,97.217950,100.000000,100.000000,7.293903,72.089495,59.997984,17.886005,52.814571,42.321531,83.429939
2,R003,Газпром Поляна,37.652812,27.178153,95.844308,80.567172,90.686087,96.041159,91.540557,69.087693,64.500308,56.071628,16.623429,54.691441,42.321531,83.429939
3,R004,Шерегеш,83.618582,83.355007,39.253523,91.033834,71.379927,98.518906,63.633493,64.839101,65.738985,21.385968,100.000000,25.481053,24.742757,63.468476
4,R005,Архыз,47.188264,33.029909,39.869439,83.578282,55.897040,94.566116,72.599951,65.583847,48.302554,27.190474,28.611971,57.646751,100.000000,12.467111


##### Проверка: линейная шкала с ограничением
* Все баллы блока в 0-100.
* Пропусков нет - региональные метрики есть у всех 32.
* Санити-чек: клип-предохранители - кто уперся в потолок роста.

In [17]:
num = scores.select_dtypes('number')
print('max:', num.max().max(), '/ min:', num.min().min())
print('NaN:', num.isna().sum().sum(), '| колонок:', scores.shape[1])

# уперлись в потолок роста (клип обрезал все, что выше +25%)
print('рост = 100 у', (scores['growth_score'] == 100).sum(), 'курортов')
print('ночи = 100 у', (scores['stay_score'] == 100).sum(), 'курортов')

max: 100.0 / min: 0.0
NaN: 0 | колонок: 16
рост = 100 у 4 курортов
ночи = 100 у 2 курортов


#### Пороговая шкала
* Цвета трасс, солнечные часы и перепад - сценарные показатели: живут в scenario_scores, в базовый рейтинг не входят. 

In [18]:
# Солнце: в витрине его нет - пристегиваем merge-ом из климатической таблицы по resort_id
climate = pd.read_csv('../data/prepared/resort_climate_features.csv')
sun = climate[['resort_id', 'winter_mean_sunshine_hours']]
scenario_scores = scenario_scores.merge(sun, on='resort_id', how='left', validate='one_to_one')

# Цвета трасс
scenario_scores['green_score'] = threshold_score(mart['green'], 10)   # семья: где учиться новичку
scenario_scores['blue_score'] = threshold_score(mart['blue'], 15)
scenario_scores['red_score'] = threshold_score(mart['red'], 10)
scenario_scores['black_score'] = threshold_score(mart['black'], 5)    # экспертные трассы

# Солнечные часы: 6 часов в зимний день
scenario_scores['sun_score'] = threshold_score(scenario_scores['winter_mean_sunshine_hours'], 6)

# Часы солнца были нужны только для балла - убираем, в таблице остаются только баллы
scenario_scores = scenario_scores.drop(columns=['winter_mean_sunshine_hours'])

# Перепад высот: порог 850 м 
scenario_scores['drop_score'] = threshold_score(mart['drop_m'], 850)

print(scenario_scores.shape)
scenario_scores.head()

(32, 9)


,resort_id,name_ru,railway_score,green_score,blue_score,red_score,black_score,sun_score,drop_score
0,R001,Роза Хутор,58.928102,100.0,100.000000,100.0,100.0,100.00000,84.705882
1,R002,Красная Поляна,92.063418,50.0,100.000000,100.0,100.0,100.00000,87.058824
2,R003,Газпром Поляна,68.924349,80.0,66.666667,100.0,80.0,100.00000,100.000000
3,R004,Шерегеш,68.924349,20.0,73.333333,80.0,20.0,99.75883,85.882353
4,R005,Архыз,0.000000,40.0,33.333333,50.0,40.0,100.00000,42.823529


##### Проверка: пороговая шкала
* Все баллы блока в 0-100.
* Солнце без пропусков - посчитано для всех 32.
* Цвета и перепад: ожидаем 6 пропусков - Кола и Аджигардак (not_collected, в сценариях скрыты).
* Санити-чек: эпичные горы по перепаду.

In [19]:
num = scenario_scores.select_dtypes('number')
print('max:', num.max().max(), '/ min:', num.min().min())
print('NaN:', num.isna().sum().sum())

# эпичные горы страны
scenario_scores.sort_values('drop_score', ascending=False).head(3)

max: 100.0 / min: 0.0
NaN: 10


,resort_id,name_ru,railway_score,green_score,blue_score,red_score,black_score,sun_score,drop_score
5,R006,Домбай,0.000000,20.0,20.000000,40.0,20.0,100.0,100.0
2,R003,Газпром Поляна,68.924349,80.0,66.666667,100.0,80.0,100.0,100.0
9,R010,Цей,0.000000,0.0,13.333333,30.0,40.0,100.0,100.0


#### Бинарная шкала
* Расчет для показателей тюбинга и фрирайда.
* Уже 0/1 - переводим в 0/100 умножением, функция не нужна.
* Пропуски ожидаем только у двух неработающих ГК.

In [20]:
scenario_scores['tubing_score'] = mart['tubing'] * 100
scenario_scores['freeride_score'] = mart['freeride'] * 100

print(scenario_scores.shape)
scenario_scores.head()

(32, 11)


,resort_id,name_ru,railway_score,green_score,blue_score,red_score,black_score,sun_score,drop_score,tubing_score,freeride_score
0,R001,Роза Хутор,58.928102,100.0,100.000000,100.0,100.0,100.00000,84.705882,100.0,100.0
1,R002,Красная Поляна,92.063418,50.0,100.000000,100.0,100.0,100.00000,87.058824,100.0,100.0
2,R003,Газпром Поляна,68.924349,80.0,66.666667,100.0,80.0,100.00000,100.000000,100.0,100.0
3,R004,Шерегеш,68.924349,20.0,73.333333,80.0,20.0,99.75883,85.882353,100.0,100.0
4,R005,Архыз,0.000000,40.0,33.333333,50.0,40.0,100.00000,42.823529,100.0,100.0


##### Проверка: бинарная шкала
* Только 0 или 100.
* Ожидаем 4 пропуска.

In [21]:
print('тюбинг = 100 у', (scenario_scores['tubing_score'] == 100).sum(), 'курортов')
print('фрирайд = 100 у', (scenario_scores['freeride_score'] == 100).sum(), 'курортов')
print('NaN:', scenario_scores[['tubing_score', 'freeride_score']].isna().sum().sum())

тюбинг = 100 у 21 курортов
фрирайд = 100 у 23 курортов
NaN: 4


### Финальная проверка: базовые показатели целиком
* Все 13 показателей + контрольный price_score_tz.
* Диапазон 0-100, пропусков нет - цена уже полечена баллом 50 выше.

In [22]:
num = scores.select_dtypes('number')
print('Размер:', scores.shape)
print('max:', num.max().max(), '/ min:', num.min().min())
print(num.isna().sum())

Размер: (32, 16)
max: 100.0 / min: 0.0
snow_days_score     0
frost_days_score    0
trips_score         0
trails_km_score     0
lifts_score         0
lodging_score       0
food_score          0
snowfall_score      0
transport_score     0
airport_score       0
price_score         0
price_score_tz      0
growth_score        0
stay_score          0
dtype: int64


### Финальная проверка: опциональные показатели целиком
* Все 9 показателей.
* Диапазон 0-100.
* 14 пропусков - по 7 у неработающих ГК.

In [23]:
num = scenario_scores.select_dtypes('number')
print('Размер:', scenario_scores.shape)
print('max:', num.max().max(), '/ min:', num.min().min())
print('NaN всего:', num.isna().sum().sum())
print(num.isna().sum())

Размер: (32, 11)
max: 100.0 / min: 0.0
NaN всего: 14
railway_score     0
green_score       2
blue_score        2
red_score         2
black_score       2
sun_score         0
drop_score        2
tubing_score      2
freeride_score    2
dtype: int64


## Итоговый рейтинг

### Функция calculate_rating
* Каждый показатель вносит вклад "балл x вес".
* Вклады складываются и делятся на 100.
* Итог - popularity_score от 0 до 100.
* Равные баллы получают одинаковое место.
* Одна функция считает все рейтинги: базовый и сценарные меняется только словарь весов.

In [24]:
def calculate_rating(score_table, weights):
    # Итог = сумма "балл x вес" по всем показателям, деленная на 100
    # (веса ТЗ - целые числа, в сумме ровно 100).
    rating = score_table.copy()

    total = 0
    for column, weight in weights.items():
        assert rating[column].between(0, 100).all(), 'Проверь баллы: ' + column
        total = total + rating[column] * weight
    rating['popularity_score'] = total / 100

    # Место 1 - лучший итог; равные итоги получают одно место (ТЗ):
    # method='min' - двоим по месту 1, следующий сразу место 3.
    rating['rank'] = rating['popularity_score'].rank(
        ascending=False, method='min'
    ).astype(int)
    return rating

In [25]:
# Мини-тест: три вымышленных курорта и три показателя.
# Цифры круглые - итог легко пересчитать на калькуляторе.
demo_scores = pd.DataFrame({
    'name_ru': ['Снежный', 'Уютный', 'Дешевый'],
    'snow_score': [100, 80, 20],
    'price_score': [40, 60, 100],
    'lift_score': [40, 60, 40],
})

demo_weights = {'snow_score': 50, 'price_score': 30, 'lift_score': 20}

demo_rating = calculate_rating(demo_scores, demo_weights)
demo_rating

,name_ru,snow_score,price_score,lift_score,popularity_score,rank
0,Снежный,100,40,40,70.0,1
1,Уютный,80,60,60,70.0,1
2,Дешевый,20,100,40,48.0,3


##### Вывод по мини-тесту
* Функция считает верно:
  * Итог Снежного проверяем на калькуляторе - 100x50 + 40x30 + 40x20 = 7000, делим на 100 - ровно 70, как в таблице.
  * Снежный и Уютный набрали поровну при разных сильных сторонах - обоим место 1, а следующий сразу место 3: места 2 нет, потому что равные баллы получают одно место (требование ТЗ, в коде - method='min').
  * Дешевый показывает роль весов: идеальная цена не спасает при слабом снеге, у которого вес 50 из 100.

### Базовый рейтинг

#### Таблица весов
* Веса - таблица ТЗ = колонка "вес" в словаре скоринга.
* Контрольная колонка price_score_tz в веса не входит: это не показатель, а проверка отклоненного режима.
* Веса в сумме дают 100.

In [26]:
tz_weights = {
    'trips_score': 25,       # поездки в регион
    'growth_score': 10,      # рост поездок
    'stay_score': 5,         # длительность поездки
    'trails_km_score': 10,   # километраж трасс
    'lifts_score': 7,        # подъемники
    'lodging_score': 5,      # размещение
    'food_score': 3,         # питание и места интереса
    'snow_days_score': 8,    # снежные дни
    'frost_days_score': 6,   # морозные дни
    'snowfall_score': 6,     # снегопад за сезон
    'airport_score': 8,      # дорога до аэропорта
    'transport_score': 2,    # местный транспорт
    'price_score': 5,        # цена ски-паса
}

assert sum(tz_weights.values()) == 100, 'Веса должны давать в сумме 100'
print('Сумма весов:', sum(tz_weights.values()))

Сумма весов: 100


#### Итоговый рейтинг

In [27]:
base_rating = calculate_rating(scores, tz_weights)

print('Размер:', base_rating.shape)
print('Итог: max', base_rating['popularity_score'].max().round(2),
      '/ min', base_rating['popularity_score'].min().round(2))
print('Пропуски в итоге и местах:',
      base_rating[['popularity_score', 'rank']].isna().sum().sum())

base_rating.sort_values('rank')[['name_ru', 'popularity_score', 'rank']]

Размер: (32, 18)
Итог: max 71.86 / min 25.52
Пропуски в итоге и местах: 0


,name_ru,popularity_score,rank
29,Сорочаны,71.860342,1
2,Газпром Поляна,69.712829,2
0,Роза Хутор,64.741943,3
1,Красная Поляна,63.394803,4
3,Шерегеш,60.002799,5
15,Горный воздух,57.879190,6
28,Игора,55.636140,7
5,Домбай,54.923213,8
4,Архыз,54.031415,9
17,Морозная гора,52.505947,10


#### Вывод о рейтинге
* Итоги собрались в узкую полосу - от 25.5 до 71.9.
* Это свойство взвешенной суммы: чтобы получить максимум, нужно быть чемпионом сразу по всем 13 показателям, а таких нет.
* Лидер - Сорочаны.
* Места идут с 1 по 32 без склеек - равных итогов не оказалось.

##### Разбор: за что курорт получил место
* Итог складывается из 13 баллов. Чтобы увидеть, какие показатели "тащат" курорт, выводим его баллы по убыванию.
* tz_weights выступает списком колонок рейтинга: берем из строки только их.

In [28]:
# Кто тащит курорт: его 13 баллов по убыванию
for resort_name in ['Сорочаны', 'Аджигардак']:
    row = base_rating[base_rating['name_ru'] == resort_name].iloc[0]
    print(resort_name, '- место', int(row['rank']),
          ', итог', round(row['popularity_score'], 1))
    # astype(float): строка таблицы хранится как object (числа вперемешку
    # с названием курорта), а round умеет только числа - превращаем явно
    print(row[list(tz_weights)].astype(float).sort_values(ascending=False).round(1).to_string())
    print()

Сорочаны - место 1 , итог 71.9
trips_score         100.0
growth_score         99.1
price_score          92.8
transport_score      77.8
snow_days_score      66.0
lifts_score          63.1
lodging_score        62.0
airport_score        60.3
frost_days_score     55.9
stay_score           55.0
food_score           54.9
trails_km_score      33.7
snowfall_score       33.6

Аджигардак - место 13 , итог 48.1
snow_days_score     73.8
frost_days_score    64.9
growth_score        59.5
transport_score     55.2
snowfall_score      51.5
price_score         50.0
trips_score         47.2
trails_km_score     46.3
lifts_score         43.2
food_score          42.8
stay_score          42.6
airport_score       26.6
lodging_score       12.5



##### Вывод по профилям
* Сорочаны берут спросом: поездки и рост почти максимальны при самом большом весе (спрос весит 40 из 100), плюс дешевый билет и доступность Подмосковья. Слабы только километраж трасс и снегопад - это склон у мегаполиса, а не снежная гора.
* Аджигардак держит зима (снег 73.8, мороз 64.9), но размещение 12.5 и аэропорт 26.6 тянут вниз, цена - нейтральные 50. 

#### Таблица достоверности
* Уверенность в данных - отдельное число рядом с рейтингом.
* Оно не входит в итог и не двигает места: у двух курортов с близким итогом качество данных может различаться.
* Формула: confidence = 50 x полнота + 25 x есть цена + 25 x точные координаты

    - полнота = заполненные обязательные поля / 11; состав полей: координаты
      (пара - одно поле), пять полей OSM (еда и места интереса - одной суммой),
      три поля климата, аэропорт, цена
    - точные координаты = точка проверена вручную (official_verified
      или checked_ok)

In [29]:
required_fields = {
    'координаты': ['latitude', 'longitude'],   # пара = одно поле
    'трассы': ['piste_total_km_6km'],
    'лифты': ['lift_lines_6km'],
    'жилье': ['lodging_objects_6km'],
    'еда+интересы': ['food_objects_6km', 'attraction_objects_6km'],
    'транспорт': ['transport_objects_6km'],
    'снежные дни': ['avg_snow_days_per_season'],
    'морозные дни': ['avg_freezing_days_per_season'],
    'снегопад': ['avg_snowfall_cm_per_season'],
    'аэропорт': ['nearest_airport_distance_km'],
    'цена': ['adult_price_min_rub'],
}

# поле заполнено, когда заполнены все его колонки
filled = pd.DataFrame({
    name: mart[cols].notna().all(axis=1) for name, cols in required_fields.items()
})

has_price = mart['adult_price_min_rub'].notna()
exact_coords = mart['coordinate_quality'].isin(['official_verified', 'checked_ok'])

base_rating['confidence_score'] = (
    50 * filled.sum(axis=1) / 11 + 25 * has_price + 25 * exact_coords
).round(2)

print('Распределение confidence_score:')
print(base_rating['confidence_score'].value_counts().to_string())

Распределение confidence_score:
confidence_score
100.00    29
45.45      2
70.45      1


In [30]:
# Кто не набрал 100: поименно, рядом с местом в рейтинге
cols = ['name_ru', 'rank', 'popularity_score', 'confidence_score']
base_rating[base_rating['confidence_score'] < 100][cols].round(2)

,name_ru,rank,popularity_score,confidence_score
11,Телецкий,30,37.23,70.45
14,КолаСпортЛэнд,23,44.09,45.45
22,Аджигардак,13,48.09,45.45


##### Вывод по достоверности
* У 29 из 32 уверенность полная - 100.
* Не добрали трое, у всех причина одна - нет разовой цены (ГК на реконструкции или закрыты), и она бьет по формуле дважды: не начисляются 25 пунктов за наличие цены, и полнота падает до 10 полей из 11.
* У Телецкого координаты проверены вручную, поэтому его итог 70.45; у Колы и Аджигардака точки не проверяли (обе не работают) - теряются еще и 25 за координаты, итог 45.45.

#### Флаги качества
* Собираем каждому курорту строку предупреждений quality_flags:
  * check coordinates (точка не проверена),
  * coordinates are proxy (точка от ближайшего объекта),
  * OSM radial overlap (радиус пересекается с соседом),
  * source error (ошибка сбора),
  * missing price (цена не найдена, балл 50),
  * не работает (closed),
  * не работает (reconstruction).

In [31]:
# Условия флагов: True = флаг ставится.
# source error не считаем - ошибок сбора не было
flag_conditions = pd.DataFrame({
    'missing price': mart['adult_price_min_rub'].isna(),
    'check coordinates': mart['coordinate_quality'] == 'manual_review_required',
    'coordinates are proxy': mart['coordinate_quality'].str.contains('proxy', na=False),
    'OSM radial overlap': mart['osm_overlap_warning'].notna(),
    'не работает (closed)': mart['operating'] == 'closed',
    'не работает (reconstruction)': mart['operating'] == 'reconstruction',
})

def collect_flags(row):
    # собираем названия флагов, у которых условие True
    active = [name for name in flag_conditions.columns if row[name]]
    return '; '.join(active) if active else 'нет'

# apply(axis=1) идет по таблице строка за строкой
base_rating['quality_flags'] = flag_conditions.apply(collect_flags, axis=1)

In [32]:
for name in flag_conditions.columns:
    print(name, ':', flag_conditions[name].sum())

cols = ['name_ru', 'rank', 'popularity_score', 'confidence_score', 'quality_flags']
base_rating[base_rating['quality_flags'] != 'нет'][cols].sort_values('rank').round(2)

missing price : 3
check coordinates : 2
coordinates are proxy : 0
OSM radial overlap : 8
не работает (closed) : 1
не работает (reconstruction) : 2


,name_ru,rank,popularity_score,confidence_score,quality_flags
2,Газпром Поляна,2,69.71,100.00,OSM radial overlap
0,Роза Хутор,3,64.74,100.00,OSM radial overlap
1,Красная Поляна,4,63.39,100.00,OSM radial overlap
22,Аджигардак,13,48.09,45.45,missing price; check coordinates; не работает ...
6,Эльбрус,14,47.49,100.00,OSM radial overlap
7,Чегет,15,47.44,100.00,OSM radial overlap
12,Большой Вудъявр,18,46.42,100.00,OSM radial overlap
14,КолаСпортЛэнд,23,44.09,45.45,missing price; check coordinates; OSM radial o...
13,Кукисвумчорр,24,42.73,100.00,OSM radial overlap
11,Телецкий,30,37.23,70.45,missing price; не работает (reconstruction)


##### Вывод по флагам
* Рекордсмен - Аджигардак: три флага сразу (нет цены, точка не проверена, реконструкция) при 13 месте - самый недостоверенный курорт верхней половины.
* Самый массовый флаг - пересечение радиусов OSM, у 8 курортов, включая лидеров.
* Для Газпрома, Розы и Красной Поляны это не дефект данных, а свойство долины: радиусы 6 км накрывают друг друга, и часть объектов вокруг считается дважды.
* Уверенность у них при этом 100 - флаг честно предупреждает, но доверия не отнимает.
* Остальные флаги собрали неработающие: Кола закрыта, Телецкий и Аджигардак в реконструкции.
* Флаги proxy и source error не сработали ни разу: proxy-точки заменены при ручной проверке координат, ошибок сбора не было.

#### Итоговая таблица базового рейтинга с оценкой уверенности в данных и флагами качества.
* Вся информация о курорте одним взглядом: итог, место, уверенность и флаги.
* Так строка будет выглядеть в итоговой таблице resort_scores и на дашборде.

In [33]:
rating_view = base_rating.sort_values('rank')[
    ['name_ru', 'popularity_score', 'rank', 'confidence_score', 'quality_flags']
].round(2)
rating_view

,name_ru,popularity_score,rank,confidence_score,quality_flags
29,Сорочаны,71.86,1,100.00,нет
2,Газпром Поляна,69.71,2,100.00,OSM radial overlap
0,Роза Хутор,64.74,3,100.00,OSM radial overlap
1,Красная Поляна,63.39,4,100.00,OSM radial overlap
3,Шерегеш,60.00,5,100.00,нет
15,Горный воздух,57.88,6,100.00,нет
28,Игора,55.64,7,100.00,нет
5,Домбай,54.92,8,100.00,нет
4,Архыз,54.03,9,100.00,нет
17,Морозная гора,52.51,10,100.00,нет


##### Вывод: итоговая таблица
* Вся глава в одной таблице: место, итог, доверие и предупреждения.
* Картина здоровая: у 29 строк уверенность 100, и флаги либо отсутствуют, либо это пересечение радиусов.
* Единственное красное пятно верхней половины - Аджигардак: 13 место при уверенности 45.45 и трех флагах. 

#### Полная проверка согласно ТЗ
* Перед сдачей рейтинга прогоняем чек-лист.
* Каждая проверка печатает результат - провалы видны сразу все, а не по одному.

In [34]:
checks = []

# 1. Веса ТЗ в сумме дают ровно 100
checks.append(('Сумма весов ТЗ = 100', sum(tz_weights.values()) == 100))

# 2. В рейтинге 32 строки и 32 уникальных курорта
checks.append(('32 строки, 32 уникальных id',
               len(base_rating) == 32 and base_rating['resort_id'].nunique() == 32))

# 3. Итог и место без пропусков
checks.append(('Итог и место без пропусков',
               base_rating[['popularity_score', 'rank']].isna().sum().sum() == 0))

# 4. Нет бесконечностей в итоге
checks.append(('Нет бесконечностей в итоге',
               not np.isinf(base_rating['popularity_score']).any()))

# 5. Итог в разумном диапазоне 0-100
checks.append(('Итог в диапазоне 0-100',
               base_rating['popularity_score'].between(0, 100).all()))

for name, ok in checks:
    print(('OK      ' if ok else 'ПРОВАЛ  ') + name)

OK      Сумма весов ТЗ = 100
OK      32 строки, 32 уникальных id
OK      Итог и место без пропусков
OK      Нет бесконечностей в итоге
OK      Итог в диапазоне 0-100


##### Вывод по полной проверке
* Пять из пяти - рейтинг принят по ТЗ: веса честные, все 32 курорта на месте, итоги без пропусков и аварий. 

#### Стресс-тест весов
* Проверяем устойчивость: что будет, если одной группе показателей придать на 20% больше важности?
* Каждой из пяти групп по очереди увеличиваем веса на 20%, остальные пропорционально сжимаем - сумма весов остается ровно 100.
* Рейтинг пересчитывается, сравниваем места с базовыми.
* Вес группы = сумма весов ТЗ входящих в нее показателей: спрос 25+10+5=40, зима 8+6+6=20, инфраструктура 10+7+5+3=25, доступность 8+2=10, цена 5.Вместе - те же 100.

In [35]:
weight_groups = {
    'спрос': ['trips_score', 'growth_score', 'stay_score'],
    'зима': ['snow_days_score', 'frost_days_score', 'snowfall_score'],
    'инфраструктура': ['trails_km_score', 'lifts_score', 'lodging_score', 'food_score'],
    'доступность': ['airport_score', 'transport_score'],
    'цена': ['price_score'],
}

# группы обязаны накрывать все 13 показателей ровно один раз
all_grouped = [col for cols in weight_groups.values() for col in cols]
assert sorted(all_grouped) == sorted(tz_weights), 'Группы должны покрывать все 13 показателей'

for group_name, cols in weight_groups.items():
    print(group_name, '-', sum(tz_weights[c] for c in cols))

спрос - 40
зима - 20
инфраструктура - 25
доступность - 10
цена - 5


In [36]:
def stress_weights(group_cols):
    # группа +20%, остальные пропорционально сжимаем, сумма остается 100
    group_sum = sum(tz_weights[c] for c in group_cols)
    other_scale = (100 - 1.2 * group_sum) / (100 - group_sum)
    weights = {}
    for col, weight in tz_weights.items():
        if col in group_cols:
            weights[col] = weight * 1.2
        else:
            weights[col] = weight * other_scale
    assert round(sum(weights.values()), 6) == 100, 'Сумма весов должна остаться 100'
    return weights

stress_ranks = pd.DataFrame({'name_ru': base_rating['name_ru'],
                             'база': base_rating['rank']})
for group_name, group_cols in weight_groups.items():
    variant = calculate_rating(scores, stress_weights(group_cols))
    stress_ranks['+' + group_name] = variant['rank'].values

stress_ranks.head(10)

,name_ru,база,+спрос,+зима,+инфраструктура,+доступность,+цена
0,Роза Хутор,3,3,3,3,3,3
1,Красная Поляна,4,4,5,4,4,4
2,Газпром Поляна,2,2,2,1,2,2
3,Шерегеш,5,6,4,5,5,5
4,Архыз,9,9,10,7,9,9
5,Домбай,8,8,7,8,8,8
6,Эльбрус,14,20,13,13,15,15
7,Чегет,15,21,14,14,16,14
8,Мамисон,31,31,31,31,31,31
9,Цей,32,32,32,32,32,32


In [37]:
# изменение мест в каждом сценарии (по модулю)
diff = (stress_ranks.drop(columns=['name_ru', 'база']) - stress_ranks[['база']].values).abs()

# худший прыжок курорта по всем пяти сценариям
jump = diff.max(axis=1)

# приклеиваем имена к прыжкам: собираем мини-таблицу из двух колонок
jump_named = pd.DataFrame({'name_ru': stress_ranks['name_ru'], 'прыжок': jump})
jump_named.sort_values('прыжок', ascending=False).head(10)

,name_ru,прыжок
6,Эльбрус,6
7,Чегет,6
20,Гора Белая,6
12,Большой Вудъявр,4
25,Абзаково,4
30,Красная Глинка,3
21,Евразия,3
4,Архыз,2
22,Аджигардак,2
23,Солнечная долина,2


##### Вывод: стресс-тест
* Голова рейтинга устойчива: четверка лидеров не меняется ни в одном из пяти сценариев, в топ-10 максимальный сдвиг - 2 места.
* Сильнее всего прыгают середина и хвост с перекошенным профилем: Эльбрус, Чегет и Гора Белая - до 6 мест, хорошая гора и зимняя погода, но без спроса и близкого аэропорта.
* Низ рейтинга стабилен при любых весах.
* Вывод: веса можно двигать в пределах 20% - картина не перевернется, спор о точных значениях весов не критичен для вершины рейтинга.

### Вывод по базовому рейтингу
* Рейтинг посчитан и принят: все 32 курорта, места 1-32 без склеек, полная проверка по ТЗ - пять из пяти.
* Итоги живут в пределах от 25.5 до 71.9 - это свойство взвешенной суммы: чемпионом сразу по всем 13 показателям быть нельзя.
* Топ-5: Сорочаны, Газпром Поляна, Роза Хутор, Красная Поляна, Шерегеш.
* Топ курорта Сорочаны обеспечивается спросом - рядом крупный город, дешевые билеты, доступность;
* У краснодарской тройки двойной источник силы - спрос региона и развитая инфраструктура;
* Шерегеш - инфраструктура, зимняя погода и цена.
* Главный сюрприз: popularity_score - это популярность, а не "лучшая гора". №1 страны - склон у мегаполиса, а не снежная вершина.
* Сценарии следующей главы покажут эту же таблицу глазами горнолыжника - картина будет другой.
* Достоверность: у 29 курортов из 32 уверенность 100. Трое без разовой цены.
* Устойчивость: стресс-тест +20% к каждой группе весов не меняет четверку лидеров ни в одном сценарии, в топ-10 сдвиг максимум 2 места. 

### Сценарный рейтинг
* Сценарный рейтинг - те же баллы, но с разными весами под разные сценарии.
* Трое персонажей:
  * хардкор-лыжник
  * семья с детьми
  * отдыхающий без лыж
* Показываем только 29 работающих курортов.

#### Веса персонажей

| Показатель | Хардкор-лыжник | Семья | Отдыхающий без лыж |
|---|---|---|---|
| Перепад высот | 25 | - | - |
| Снежные дни | 20 | - | - |
| Черные трассы | 15 | - | - |
| Фрирайд | 10 | - | - |
| Снегопад | 10 | - | - |
| Подъемники | 10 | - | - |
| Красные трассы | 10 | - | - |
| Зеленые трассы | - | 25 | - |
| Тюбинг | - | 20 | - |
| ЖД-станция | - | 20 | - |
| Цена | - | 15 | 15 |
| Питание и места интереса | - | 10 | 25 |
| Размещение | - | 5 | 20 |
| Аэропорт | - | 5 | 15 |
| Солнечные часы | - | - | 25 |

Сумма весов каждого персонажа - 100. Базовые показатели из словаря выше
участвуют у персонажей по мере нужд человека.

In [38]:
scenario_matrix = scores.merge(scenario_scores, on=['resort_id', 'name_ru'],
                               validate='one_to_one')
scenario_matrix = scenario_matrix.merge(mart[['resort_id', 'operating']],
                                        on='resort_id', validate='one_to_one')
scenario_matrix = scenario_matrix[scenario_matrix['operating'] == 'yes'] \
    .drop(columns='operating').reset_index(drop=True)

print('Работающих курортов в сценариях:', len(scenario_matrix))
print('Пропусков:', int(scenario_matrix.isna().sum().sum()))

Работающих курортов в сценариях: 29
Пропусков: 0


#### Персонаж 1. Хардкор-лыжник
Ездит за снегом и рельефом: перепад, черные трассы, внетрассовое. Цена и дорога почти не важны.

In [39]:
hardcore_weights = {
    'drop_score': 25,        # перепад - главный экстрим
    'snow_days_score': 20,   # снег должен быть
    'black_score': 15,       # черные трассы
    'freeride_score': 10,    # внетрассовое катание
    'snowfall_score': 10,    # снегопад за сезон
    'lifts_score': 10,       # подъемники
    'red_score': 10,         # красные трассы
}
assert sum(hardcore_weights.values()) == 100

hardcore_rating = calculate_rating(scenario_matrix, hardcore_weights)
hardcore_rating.sort_values('rank')[['name_ru', 'popularity_score', 'rank']].head(5)

,name_ru,popularity_score,rank
7,Чегет,88.670376,1
2,Газпром Поляна,80.507940,2
11,Большой Вудъявр,76.804390,3
5,Домбай,76.611709,4
10,Манжерок,76.505873,5


##### Вывод: хардкор-лыжник
Лидер - Чегет, 88.7: перепады высоты, черные и фрирайд -
все по 100. В базовом рейтинге он 15-й. Сорочаны из топа исчезают совсем:
мегаполис не помогает, когда нужны перепад и черные трассы. 

#### Персонаж 2. Семья с детьми
Новичковые зеленые трассы, тюбинг, поезд вместо самолета, бюджет
и где поесть с детьми.

In [40]:
family_weights = {
    'green_score': 25,       # зеленые трассы для новичков
    'tubing_score': 20,      # тюбинг
    'railway_score': 20,     # поезд без перелета с детьми
    'price_score': 15,       # бюджет
    'food_score': 10,        # поесть с детьми
    'lodging_score': 5,
    'airport_score': 5,
}
assert sum(family_weights.values()) == 100

family_rating = calculate_rating(scenario_matrix, family_weights)
family_rating.sort_values('rank')[['name_ru', 'popularity_score', 'rank']].head(5)

,name_ru,popularity_score,rank
2,Газпром Поляна,73.038079,1
0,Роза Хутор,72.006591,2
1,Красная Поляна,71.595483,3
13,Горный воздух,71.391290,4
27,Красная Глинка,69.729836,5


##### Вывод: семья
Краснодарская тройка впереди: зеленые трассы, тюбинг, олимпийская ЖД-ветка,
еда вокруг. Горный воздух и Красная Глинка рядом - тюбинг и поезд.
Базовый лидер Сорочаны здесь только в середине: зеленых мало, тюбинга нет.

#### Персонаж 3. Отдыхающий без лыж
Едет в горы не кататься: акцент на солнце, хорошей еде и жилье.
Трассы не важны совсем - в весах нет ни одного трассового показателя.

In [41]:
comfort_weights = {
    'sun_score': 25,        # солнышко и тепло - главное
    'food_score': 25,       # хорошая еда вокруг
    'lodging_score': 20,    # комфортное жилье
    'price_score': 15,
    'airport_score': 15,    # добраться без подвига
}
assert sum(comfort_weights.values()) == 100

comfort_rating = calculate_rating(scenario_matrix, comfort_weights)
comfort_rating.sort_values('rank')[['name_ru', 'popularity_score', 'rank']].head(5)

,name_ru,popularity_score,rank
13,Горный воздух,90.666793,1
1,Красная Поляна,81.682598,2
3,Шерегеш,78.759757,3
2,Газпром Поляна,77.997630,4
27,Красная Глинка,72.464745,5


##### Вывод: отдыхающий без лыж
Лидер - Горный воздух, 90.7: солнечный Сахалин,
человеческие цены, еда вокруг. Вторая - Красная Поляна: кавказское солнце
и кухня. Сорочаны, базовый лидер, - только 10-е место. Ни одного трассового показателя в весах -
и картина перевернулась полностью.

In [42]:
summary = pd.DataFrame({
    'база': base_rating.sort_values('rank')['name_ru'].head(5).tolist(),
    'хардкор': hardcore_rating.sort_values('rank')['name_ru'].head(5).tolist(),
    'семья': family_rating.sort_values('rank')['name_ru'].head(5).tolist(),
    'комфорт': comfort_rating.sort_values('rank')['name_ru'].head(5).tolist(),
})
summary

,база,хардкор,семья,комфорт
0,Сорочаны,Чегет,Газпром Поляна,Горный воздух
1,Газпром Поляна,Газпром Поляна,Роза Хутор,Красная Поляна
2,Роза Хутор,Большой Вудъявр,Красная Поляна,Шерегеш
3,Красная Поляна,Домбай,Горный воздух,Газпром Поляна
4,Шерегеш,Манжерок,Красная Глинка,Красная Глинка


### Вывод по сценарным рейтингам
Один и тот же набор баллов дает четыре разные картины: база - Сорочаны,
хардкор-лыжник - Чегет (15-й в базе), семья - Газпром, отдыхающий без лыж -
Горный воздух. Лидер меняется вместе
с разными сценариями - это и есть ответ на вопрос "зачем сценарии": базовый рейтинг
показывает среднюю температуру, вымышленные персонажи - конкретные потребности.
Сценарий ТЗ "большой вес климата" закрыт персонажем хардкор-лыжника,
"большой вес доступности" посчитан в стресс-тесте (колонка "+доступность"
таблицы мест).

## Ответы на вопросы для исследования согласно ТЗ
* Вопросы из ТЗ к анализу рейтинга. Ответы собраны из расчетов выше, ссылки ведут в разделы, где они считались. 

### Вопрос 1. Какие три показателя сильнее всего влияют на итоговый порядок?
Влияние меряем ранговой корреляцией: сравниваем порядок курортов по показателю с порядком по итогу. Корреляция 1 - показатель полностью определяет итог, 0 - порядок показателя не связан с итоговым, минус - тянет в обратную сторону.

In [43]:
influence = pd.Series({
    col: scores[col].rank().corr(base_rating['popularity_score'].rank())
    for col in tz_weights
})
influence.sort_values(ascending=False).round(2)

lifts_score         0.71
food_score          0.70
lodging_score       0.66
transport_score     0.59
trips_score         0.49
airport_score       0.44
growth_score        0.42
trails_km_score     0.39
stay_score          0.36
snowfall_score      0.22
price_score         0.03
snow_days_score    -0.08
frost_days_score   -0.36
dtype: float64

**Ответ:** три самых влияющих показателя - подъемники (0.71), питание
и места интереса (0.70), размещение (0.66). Вся тройка - инфраструктура.
Не поездки, хотя у них самый большой вес (25): у них корреляция 0.49.
Причина: спрос региональный - у курортов одного
региона одинаковый балл поездок, поэтому спрос плохо различает курорты.
А инфраструктура различает тонко: у соседей по долине лифты, еда и жилье
разные. К тому же инфраструктурные показатели идут согласованно -
крупные комплексы впереди по всем показателям сразу. Морозные дни - единственный показатель с минусом (-0.36):
самые морозные курорты стоят внизу - мороз без спроса и инфраструктуры
не спасает положение.

### Вопрос 2. Какие курорты находятся высоко из-за регионального спроса, а какие - благодаря климату или инфраструктуре?
* Вклад группы в итог курорта = сумма "балл x вес" ее показателей / 100.
* Вклады пяти групп складываются ровно в popularity_score - можно разложить итог любого курорта по группам.

In [44]:
group_contrib = pd.DataFrame({'name_ru': base_rating['name_ru']})
for group_name, group_cols in weight_groups.items():
    group_contrib[group_name] = sum(
        scores[col] * tz_weights[col] for col in group_cols) / 100

group_contrib['место'] = base_rating['rank'].values
view = group_contrib.set_index('name_ru').sort_values('место').round(1)
view

,спрос,зима,инфраструктура,доступность,цена,место
name_ru,,,,,,
Сорочаны,37.7,10.7,12.5,6.4,4.6,1
Газпром Поляна,32.4,8.8,22.0,5.8,0.8,2
Роза Хутор,32.4,3.0,23.5,5.9,0.0,3
Красная Поляна,32.4,0.4,23.5,6.2,0.9,4
Шерегеш,15.5,15.6,20.9,3.0,5.0,5
Горный воздух,14.1,13.5,15.9,9.8,4.6,6
Игора,27.3,10.2,9.9,4.3,3.9,7
Домбай,20.6,16.7,15.0,1.9,0.7,8
Архыз,20.6,9.7,19.2,3.1,1.4,9


**Ответ:** главная группа - источник итога - делит топ-10 так: спрос - семеро
(Сорочаны 37.7, краснодарская тройка по 32.4, Игора 27.3, Домбай и Архыз
по 20.6), инфраструктура - двое (Шерегеш 20.9, Горный воздух 15.9),
зима - одна (Морозная гора 17.8). Спрос у краснодарской тройки одинаковый
(регион один), поэтому их порядок внутри топа определили зима и цена.
Эльбрус и Чегет - климатические горы, но спрос региона мал, поэтому климат
держит их в середине, а не в топе.

### Вопрос 3. У каких объектов высокий индекс сочетается с низкой уверенностью?

**Ответ:** Аджигардак - 13 место при уверенности 45.45 и трех флагах
(нет разовой цены, точка не проверена, реконструкция). Единственный курорт
верхней половины с неполными данными. КолаСпортЛэнд тоже с 45.45,
но она на 23 месте - вне "высокого индекса".
Считалось в разделе "Таблица достоверности".

### Вопрос 4. Какие места меняются при увеличении каждого группового веса на 20 процентов?

**Ответ:** вершина стоит: четверка лидеров не меняется ни в одном из пяти
сценариев, в топ-10 максимальный сдвиг - 2 места. Сильнее всех прыгают
середина и хвост с перекошенным профилем: Эльбрус, Чегет и Гора Белая -
до 6 мест (при +20% спроса Эльбрус падает с 14 на 20, Чегет с 15 на 21).
Лидер меняется ровно при одном условии: +20% инфраструктуры - и Газпром
обходит Сорочаны. Низ рейтинга стабилен при любых весах.
Считалось в разделе "Стресс-тест весов".

### Вопрос 5. Есть ли пары соседних курортов, для которых OSM считает одни и те же объекты?

**Ответ:** да, семь пар - восемь курортов с флагом OSM radial overlap:
краснодарская тройка (Роза Хутор, Газпром, Красная Поляна - расстояния
4.8-6.6 км при радиусе сбора 6 км), пара Эльбрус-Чегет (5.7 км)
и кировская группа Большой Вудъявр - Кукисвумчорр - КолаСпортЛэнд
(3.1-7.5 км). Часть объектов вокруг у них считается дважды -
флаг честно предупреждает об этом, не отнимая доверия.
Считалось в разделе "Флаги качества".

### Вопрос 6. Как расширенные показатели Росстата меняют первые десять мест?

**Ответ:** слой не подключали - ТЗ (раздел "Дополнительный слой Росстата")
прямо называет его необязательным для первой версии рейтинга. Для него нужны
данные КСР по регионам: размещённые туристы, выручка и зимняя доля
размещений. Это задел на развитие: при появлении данных показатели
добавляются в витрину и в веса без перестройки рейтинга.

### Вопрос 7. Каких данных не хватает, чтобы говорить о фактическом спросе на отдельный курорт?

**Ответ:** все наши данные спроса - региональные: Росстат считает субъекты,
курортных рядов не существует, и три курорта одного региона получают
одинаковые значения. Для фактического спроса
на конкретный курорт не хватает: продаж ски-пассов, загрузки отелей
у склона, билетов до ближайшей станции или аэропорта, поисковых запросов
и бронирований по названию курорта.